In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('images', exist_ok=True)

# ══════════════════════════════════════════════════════════════════
#  PHASE 1 — LOAD, CLEAN & MERGE BOTH DATASETS
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 1: LOADING, CLEANING & MERGING DATASETS")
print("="*65)

# ──────────────────────────────────────────────────────────────────
# 1A ── VISSER PLANT REPORT
#   Raw shape  : 12 rows × 48 columns
#   Date col   : '25-Jan' … '26-Feb'  (YY-Mon format)
#   Day cols   : '01' … '31'  →  daily kWh yield per inverter
# ──────────────────────────────────────────────────────────────────
print("\n📂 Loading Visser_plant_report.csv ...")
visser_raw = pd.read_csv('Visser_plant_report.csv')

# ── FIX 1: Strip & standardise column names ──────────────────────
visser_raw.columns = visser_raw.columns.str.strip()

print(f"   Raw shape  : {visser_raw.shape}")
print(f"   Duplicates : {visser_raw.duplicated().sum()} rows removed")
visser_raw = visser_raw.drop_duplicates()

# ── FIX 2: Drop entirely-empty 'Owner' column (all 12 NaN) ───────
owner_nulls = visser_raw['Owner'].isnull().sum()
print(f"   'Owner' col: {owner_nulls}/12 values are NaN → column dropped")
visser_raw = visser_raw.drop(columns=['Owner'])

# ── FIX 3: Parse 'This Month Earning' → clean numeric (CAD) ──────
# Values look like '23.200000CAD' — strip currency, convert to float
visser_raw['earning_cad'] = (
    visser_raw['This Month Earning']
    .str.replace('CAD', '', regex=False)
    .str.strip()
    .astype(float)
)
print(f"   'This Month Earning' cleaned → 'earning_cad' (float, CAD)")

# ── FIX 4: Flag 'Current Status' — plant is OFFLINE for all rows ─
status_vals = visser_raw['Current Status'].unique()
print(f"   'Current Status' values: {status_vals} "
      f"— plant is offline; kept as reference flag")

# ── FIX 5: Parse the Date column correctly ────────────────────────
# '25-Jan' → 2025-01-01,  '26-Feb' → 2026-02-01
def parse_visser_month(date_str):
    try:
        yy, mon = date_str.strip().split('-')
        return pd.to_datetime(f"01-{mon}-20{yy}", format='%d-%b-%Y')
    except Exception:
        return pd.NaT

visser_raw['month_parsed'] = visser_raw['Date'].apply(parse_visser_month)
bad_dates = visser_raw['month_parsed'].isna().sum()
if bad_dates:
    print(f"   ⚠ {bad_dates} unparseable dates dropped")
    visser_raw = visser_raw.dropna(subset=['month_parsed'])

# ── FIX 6: Melt day columns → long format ────────────────────────
day_cols = [str(i).zfill(2) for i in range(1, 32)]
day_cols  = [c for c in day_cols if c in visser_raw.columns]

visser_long = visser_raw.melt(
    id_vars=['month_parsed', 'Inverter Number', 'Current Status',
             'Installed Capacity (kWp)', 'Monthly Yield (kWh)',
             'This Month Full Load Hours (h)', 'earning_cad'],
    value_vars=day_cols,
    var_name='day_num',
    value_name='daily_yield_kwh'
)

# Build proper calendar date  (invalid calendar dates e.g. Feb-30 → NaT)
visser_long['date'] = pd.to_datetime(
    visser_long['month_parsed'].dt.year.astype(str) + '-' +
    visser_long['month_parsed'].dt.month.astype(str).str.zfill(2) + '-' +
    visser_long['day_num'],
    errors='coerce'
)

# ── FIX 7: Log & remove NaN yield rows (missing days e.g. Feb-30) ─
nan_yield  = visser_long['daily_yield_kwh'].isna().sum()
nan_date   = visser_long['date'].isna().sum()
print(f"   Day NaN yields   : {nan_yield} rows dropped "
      f"(months shorter than 31 days)")
print(f"   Invalid calendar : {nan_date} rows dropped "
      f"(e.g. Feb-30, Apr-31)")

visser_long = visser_long.dropna(subset=['date', 'daily_yield_kwh'])

# ── FIX 8: Log & remove zero-yield rows ──────────────────────────
zero_yield = (visser_long['daily_yield_kwh'] == 0).sum()
print(f"   Zero-yield days  : {zero_yield} rows removed "
      f"(no production recorded)")
visser_long = visser_long[visser_long['daily_yield_kwh'] > 0].copy()

# ── FIX 9: Remove negative yield (physically impossible) ─────────
neg_yield = (visser_long['daily_yield_kwh'] < 0).sum()
if neg_yield:
    print(f"   Negative yields  : {neg_yield} rows removed")
    visser_long = visser_long[visser_long['daily_yield_kwh'] >= 0]

# ── Assign clean column names ─────────────────────────────────────
visser_long['source']          = 'Visser'
visser_long['unit_id']         = ('Visser-INV-' +
                                   visser_long['Inverter Number'].astype(str))
visser_long['installed_kw']    = visser_long['Installed Capacity (kWp)']
visser_long['full_load_hours'] = visser_long['This Month Full Load Hours (h)']
visser_long['monthly_yield']   = visser_long['Monthly Yield (kWh)']
visser_long['plant_status']    = visser_long['Current Status']
visser_long['earning_cad']     = visser_long['earning_cad']

visser_df = visser_long[[
    'date', 'unit_id', 'source', 'daily_yield_kwh',
    'installed_kw', 'full_load_hours', 'monthly_yield',
    'plant_status', 'earning_cad'
]].copy()

print(f"\n   ✅ Visser clean : {len(visser_df):>4} daily records "
      f"| Units : {visser_df['unit_id'].unique().tolist()}")

# ──────────────────────────────────────────────────────────────────
# 1B ── BISSELL THRIFT 118 AVE
#   Raw shape  : 136 rows × 8 columns (daily, Aug 2025 – Jan 2026)
#   Row 0      : unit-header labels ('[kWh]', '[kWh/kWp]') → skip
#   Date col   : dd.MM.yyyy  e.g. '20.08.2025'
#   Inv cols   : 3 Fronius Primo 7.6-1 inverters
# ──────────────────────────────────────────────────────────────────
print("\n📂 Loading Bissell_Thrift_118_Ave_01012025-01012026 (1).xlsx ...")
bissell_raw = pd.read_excel(
    'Bissell_Thrift_118_Ave_01012025-01012026 (1).xlsx',
    skiprows=1          # row 0 = '[kWh]' unit headers — not data
)
bissell_raw.columns = [
    'date',
    'inv1_kwh', 'inv2_kwh', 'inv3_kwh',
    'inv1_kwh_per_kwp', 'inv2_kwh_per_kwp', 'inv3_kwh_per_kwp',
    'total_kwh'
]
print(f"   Raw shape  : {bissell_raw.shape}")

# ── FIX 10: Parse date column (format dd.MM.yyyy) ─────────────────
bissell_raw['date'] = pd.to_datetime(
    bissell_raw['date'], format='%d.%m.%Y', errors='coerce'
)
bad_dates_b = bissell_raw['date'].isna().sum()
if bad_dates_b:
    print(f"   ⚠ {bad_dates_b} unparseable dates dropped")
bissell_raw = bissell_raw.dropna(subset=['date'])

# ── FIX 11: Force all energy cols to numeric ──────────────────────
energy_cols = [
    'inv1_kwh', 'inv2_kwh', 'inv3_kwh',
    'inv1_kwh_per_kwp', 'inv2_kwh_per_kwp', 'inv3_kwh_per_kwp',
    'total_kwh'
]
for col in energy_cols:
    bissell_raw[col] = pd.to_numeric(bissell_raw[col], errors='coerce').fillna(0)

# ── FIX 12: Remove duplicate dates ───────────────────────────────
dups = bissell_raw.duplicated(subset=['date']).sum()
print(f"   Duplicates : {dups} rows removed")
bissell_raw = bissell_raw.drop_duplicates(subset=['date'])

# ── FIX 13: Remove negative values ───────────────────────────────
for col in ['inv1_kwh', 'inv2_kwh', 'inv3_kwh']:
    neg = (bissell_raw[col] < 0).sum()
    if neg:
        print(f"   ⚠ Negative {col}: {neg} values set to 0")
        bissell_raw[col] = bissell_raw[col].clip(lower=0)

# ── FIX 14: Verify inverter sum matches total ─────────────────────
bissell_raw['inv_sum'] = (
    bissell_raw['inv1_kwh'] + bissell_raw['inv2_kwh'] + bissell_raw['inv3_kwh']
)
mismatch = (abs(bissell_raw['inv_sum'] - bissell_raw['total_kwh']) > 0.5).sum()
print(f"   Inverter sum vs total mismatch: {mismatch} rows "
      f"({'✅ data is consistent' if mismatch == 0 else '⚠ check data'})")

# ── FIX 15: Log zero-production rows before dropping ─────────────
zero_rows_b = (bissell_raw['total_kwh'] == 0).sum()
print(f"   Zero-production days: {zero_rows_b} rows removed "
      f"(no generation — nighttime/offline)")

# ── Melt 3 inverter columns → long format ────────────────────────
bissell_long = bissell_raw.melt(
    id_vars=['date', 'total_kwh'],
    value_vars=['inv1_kwh', 'inv2_kwh', 'inv3_kwh'],
    var_name='inv_raw',
    value_name='daily_yield_kwh'
)
bissell_long['unit_id'] = bissell_long['inv_raw'].map({
    'inv1_kwh': 'Bissell-INV-1',
    'inv2_kwh': 'Bissell-INV-2',
    'inv3_kwh': 'Bissell-INV-3'
})
bissell_long['source']          = 'Bissell'
# Each Fronius Primo 7.6-1 208-240 inverter has a 7.6 kW nameplate rating
bissell_long['installed_kw']    = 7.6
bissell_long['full_load_hours'] = np.nan
bissell_long['monthly_yield']   = np.nan
bissell_long['plant_status']    = 'Active'
bissell_long['earning_cad']     = np.nan

# Keep only rows with actual production
bissell_long = bissell_long[bissell_long['daily_yield_kwh'] > 0].copy()

bissell_df = bissell_long[[
    'date', 'unit_id', 'source', 'daily_yield_kwh',
    'installed_kw', 'full_load_hours', 'monthly_yield',
    'plant_status', 'earning_cad'
]].copy()

print(f"\n   ✅ Bissell clean: {len(bissell_df):>4} daily records "
      f"| Units : {bissell_df['unit_id'].unique().tolist()}")

# ──────────────────────────────────────────────────────────────────
# 1C ── MERGE & FINAL COMBINED CLEANING
# ──────────────────────────────────────────────────────────────────
combined = pd.concat([visser_df, bissell_df], ignore_index=True)
combined  = combined.sort_values(['unit_id', 'date']).reset_index(drop=True)

# Final combined dedup check
combined_dups = combined.duplicated(subset=['date','unit_id']).sum()
if combined_dups:
    print(f"   ⚠ {combined_dups} duplicate unit-date rows in combined set → removed")
    combined = combined.drop_duplicates(subset=['date','unit_id'])

# ── DATA QUALITY REPORT ──────────────────────────────────────────
print(f"""
┌─────────────────────────────────────────────────────┐
│              DATA QUALITY REPORT                    │
├──────────────────────────┬──────────────────────────┤
│  Visser clean records    │  {len(visser_df):>6}                  │
│  Bissell clean records   │  {len(bissell_df):>6}                  │
│  Combined total          │  {len(combined):>6}                  │
│  Units monitored         │  {combined['unit_id'].nunique():>6}                  │
│  Date range              │  {str(combined['date'].min().date())} → {str(combined['date'].max().date())}  │
│  Missing yield values    │  {combined['daily_yield_kwh'].isna().sum():>6}                  │
│  Negative yield values   │  {(combined['daily_yield_kwh'] < 0).sum():>6}                  │
│  Remaining duplicates    │  {combined.duplicated(subset=['date','unit_id']).sum():>6}                  │
└──────────────────────────┴──────────────────────────┘
""")
